# 09 · Deterministic steps and typed variables

## Goal

Build the first version of the renewal-check workflow: a manual trigger,
typed input variables, a couple of deterministic steps, and output feeding
the next step's input. Learn where designer testing is free and where it
stops being free.


## Prereqs

Asserted below, not just stated — this cell fails loudly if a prior notebook's step wasn't actually completed.


In [ ]:
from csx.config import load_settings
from csx.clients import get_copilot_client
settings = load_settings()
client = get_copilot_client(settings, delegated=True)


## Concept

Workflows are the deterministic complement to the agent's reasoning:
typed inputs and outputs, explicit steps, no "the model decided to phrase
it differently this time." Use them wherever the *procedure* is fixed even
if the inputs vary — checking a renewal's spend and performance signals is
a fixed three-step lookup regardless of which supplier triggered it.

**Worth knowing before you start iterating:** workflow testing in the
designer does not meter Copilot Credits — only live, executed actions from
a real conversation do. That makes the designer the cheap place to iterate
on step logic; save your credit budget for the `run_suite()` calls that
actually exercise the published agent.


## Build


### workflows/renewal-check.yaml — linear v1


In [ ]:
import yaml
from pathlib import Path
workflow_dir = Path("../agents/contract-renewal-desk/workflows")
workflow_dir.mkdir(exist_ok=True)

workflow = {
    "name": "renewal-check",
    "trigger": {"type": "manual"},
    "inputs": [
        {"name": "supplierName", "type": "string", "required": True},
    ],
    "steps": [
        {
            "id": "lookupSpend",
            "type": "dataverse-query",
            "entity": "crd_supplierspend",
            "filter": "supplierName eq @{inputs.supplierName}",
            "output": "spendRecord",
        },
        {
            "id": "lookupPerformance",
            "type": "dataverse-query",
            "entity": "crd_supplierperformance",
            "filter": "supplierName eq @{inputs.supplierName}",
            "output": "performanceRecord",
        },
        {
            "id": "summarise",
            "type": "compose",
            # output -> input chaining: this step consumes both prior outputs directly
            "input": "Spend: @{steps.lookupSpend.output.spendRecord}, Performance: @{steps.lookupPerformance.output.performanceRecord}",
            "output": "summary",
        },
    ],
    "outputs": [{"name": "summary", "value": "@{steps.summarise.output.summary}"}],
}
(workflow_dir / "renewal-check.yaml").write_text(yaml.dump(workflow, sort_keys=False))
print((workflow_dir / "renewal-check.yaml").read_text())


In [ ]:
from csx.pac import copilot_push
import subprocess
copilot_push(Path("../agents/contract-renewal-desk"))
subprocess.run(["pac", "copilot", "publish", "--name", "crd_contract-renewal-desk"], check=True)


### Test in the designer first (free) before invoking live (metered)


Open the workflow in the designer, run **Test** with `supplierName=Meridian Cables`, confirm `summary` populates from both lookups before moving to the Verify cell below.


## Verify

Same harness, same golden set, every notebook.


In [ ]:
from csx.verify import run_suite, load_golden
from csx.cost import CreditMeter
meter = CreditMeter(environment_id=settings.get("DATAVERSE_ENV_ID"))

cases = load_golden(tags=["core"])
suite = run_suite(client, cases=cases, credit_meter=meter, min_pass_rate=0.8)
# workflow-tagged cases wait for 10's branching — this notebook only proves the linear chain compiles and runs


## Cost


In [ ]:
meter.report_cost("09", budget=settings.get("COPILOT_CREDIT_BUDGET"), delta_credits=suite.total_credits, note="designer testing free; only the core-suite live invocations here meter")


## Teardown


In [ ]:
print("No teardown — renewal-check.yaml persists and grows in 10-11.")
